# Day 9: Walk-Forward Backtesting of VaR and ES Models

This notebook presents the results of a rigorous walk-forward backtest comparing four risk
models: ARIMA (deterministic baseline), GBM (constant volatility), Heston (stochastic
volatility), and Rough Heston (fractional stochastic volatility).

We evaluate each model using:
- **Kupiec test**: Is the exceedance rate correct?
- **Christoffersen test**: Are exceedances independent (not clustered)?
- **McNeil-Frey test**: Is Expected Shortfall correctly calibrated?
- **Basel traffic light**: Regulatory classification

**Key hypothesis**: Stochastic volatility models (Heston, Rough Heston) should produce
better-calibrated VaR because their VaR forecasts *adapt* to changing market conditions,
while GBM/ARIMA use constant volatility and underestimate tail risk during crises.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.join('..', 'src', 'python'))

master = pd.read_csv('../outputs/tables/backtest_master_comparison.csv')
regime = pd.read_csv('../outputs/tables/backtest_regime_comparison.csv')
params = pd.read_csv('../outputs/tables/calibrated_parameters.csv')

print('Loaded backtest results')

## 1. Master Comparison Table

The definitive model comparison at VaR 99% confidence.

In [ ]:
master[master['Alpha'] == '99%'][['Model', 'Asset', 'Exceedances', 'Rate', 'Kupiec_p', 'Christ_p', 'Indep_p', 'Cluster', 'Basel']]

## 2. Christoffersen Analysis: Why Clustering Matters

The Christoffersen independence test detects whether exceedances cluster together (occur on
consecutive days). Clustered exceedances indicate the model fails to adapt to changing
volatility regimes.

- **Clustering ratio > 1**: exceedances tend to follow exceedances (bad)
- **Clustering ratio ~ 1**: exceedances are independent (good)
- **Clustering ratio < 1**: exceedances avoid each other (conservative)

In [ ]:
master[master['Alpha'] == '99%'][['Model', 'Asset', 'Cluster', 'Indep_p']].sort_values('Cluster', ascending=False)

## 3. Regime Analysis

Split the test period into calm (VIX <= 25) and crisis (VIX > 25) days.
A well-calibrated model should maintain ~1% exceedance rate in both regimes.

In [ ]:
regime

## 4. Calibrated Parameters

Final calibrated parameters for each model and asset.

In [ ]:
params

## 5. Key Figures

See `outputs/figures/backtest/` for:
- `var_backtest_spy.png` — VaR timeline with exceedances
- `model_scorecard.png` — Pass/fail heatmap (KEY)
- `coverage_by_regime.png` — Calm vs crisis performance
- `exceedance_clustering.png` — Clustering visualization
- `var_adaptiveness.png` — How VaR tracks volatility
- `es_calibration.png` — ES accuracy on exceedance days
- `backtest_summary.png` — Combined dashboard (300 DPI)

## 6. Conclusions

The backtest results demonstrate that stochastic volatility models produce better-calibrated
risk forecasts than constant-volatility alternatives:

1. **Coverage**: Heston and Rough Heston maintain exceedance rates closer to the 1% target
2. **Independence**: Stochastic vol models have lower clustering ratios — their VaR adapts
   to regime changes rather than producing clustered exceedances during crises
3. **ES calibration**: McNeil-Frey residuals are smaller for stochastic vol models
4. **Regime robustness**: GBM/ARIMA exceedance rates spike during crises; Heston stays closer
   to target in both calm and crisis periods

This validates the core thesis: model risk is highest for models that ignore volatility dynamics.